# LedgerLock Setup Guide

This notebook will guide you through setting up and running the LedgerLock AP invoice fraud detection POC. We'll:
1. Verify the Python environment
2. Install required dependencies
3. Run a sample fraud detection pipeline
4. Visualize the results

## 1. Check Python Installation

In [ ]:
import sys
print(f"Python version: {sys.version}")

# Required packages for LedgerLock
required_packages = [
    'pandas',
    'pdfplumber',
    'openpyxl',
    'rapidfuzz',
    'scikit-learn',
    'plotly'
]

import pkg_resources
installed = {pkg.key for pkg in pkg_resources.working_set}
print("\nRequired packages status:")
for package in required_packages:
    print(f"{package}: {'✓' if package in installed else '✗'}")

## 2. Load and Preview Sample Data

In [ ]:
from src import parser

# Load sample data
invoices = parser.load_invoices('../data/invoices_sample.csv')
vendors = parser.load_vendors('../data/vendor_master.csv')
bols = parser.load_bols('../data/bol_sample.csv')

print("Sample invoices:")
display(invoices.head())
print("\nVendor master:")
display(vendors)

## 3. Run Fraud Detection Rules

In [ ]:
from src import rules
import pandas as pd

# Run all fraud checks
flags = [
    rules.flag_duplicates(invoices),
    rules.flag_bank_mismatch(invoices, vendors),
    rules.flag_unknown_vendor(invoices, vendors),
    rules.flag_unusual_amounts(invoices, vendors),
    rules.flag_bol_mismatch(invoices, bols)
]

all_flags = pd.concat(flags, ignore_index=True)
print(f"Found {len(all_flags)} potential fraud flags:")
display(all_flags)

## 4. Visualize Results

In [ ]:
import plotly.express as px

# Create flag counts visualization
flag_counts = all_flags['reason'].value_counts().reset_index()
flag_counts.columns = ['Flag Type', 'Count']

fig = px.bar(flag_counts, 
            x='Flag Type', 
            y='Count',
            title='AP Invoice Fraud Flags by Type')
fig.show()

# Save results
from src import report
report.export_report(all_flags, '../data/fraud_report.xlsx')
print("\nDetailed fraud report saved to data/fraud_report.xlsx")

## Next Steps

Now that we have the basic fraud detection pipeline working, you can:

1. Add more sample data to test edge cases
2. Implement additional fraud rules (e.g., weekend anomalies, accessorial charges)
3. Tune the thresholds for unusual amounts and fuzzy matching
4. Add PDF invoice parsing support
5. Build a web dashboard using FastAPI + React

For the full source code and documentation, visit the LedgerLock repository.